# ITO5201 – Assessment 1: Section 1
## Model Complexity and Model Selection
**Student:** Johannes Coetzee  
**Student Number:** 36384852

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator

DEBUG = True

---
## Question 1 – KNN Regressor
### Q1.I – Implement `KnnRegressor`
*GitHub issues: #1, #2, #3*

In [ ]:
class KnnRegressor(BaseEstimator):
    def __init__(self, k=5):
        self.k = k

    def fit(self, x, y):
        self.x_train_ = x
        self.y_train_ = y
        return self

    def predict(self, x):
        x = np.array(x)
        diffs = x[:, np.newaxis, :] - self.x_train_[np.newaxis, :, :]
        distances = np.sqrt((diffs ** 2).sum(axis=2))
        neighbour_idx = np.argpartition(distances, self.k, axis=1)[:, :self.k]
        return self.y_train_[neighbour_idx].mean(axis=1)
    


### Q1.II – Evaluate on `diabetes` and `california_housing`
*GitHub issues: #4, #5*

In [10]:
from sklearn.datasets import load_diabetes, fetch_california_housing

# Load datasets
diabetes = load_diabetes()
california = fetch_california_housing()

# Switched this function to RNG.permutation for better randomness and performance
# Elimante the need for RNG.shuffle and slicing, which is O(N) and O(1) respectively, 
# instead of using np.random.choice which is O(N log N)
def train_test_split(x, y, train_size=0.6, random_state=None):
    RNG = np.random.default_rng(random_state)
    N = len(x)
    N_train = round(N * train_size)
    idx = RNG.permutation(N)          # O(N) Fisher-Yates shuffle
    idx_train = idx[:N_train]         # O(1) slice
    idx_test  = idx[N_train:]         # O(1) slice
    return x[idx_train], x[idx_test], y[idx_train], y[idx_test]


In [11]:
"""
Functions to compute Sum of Squared Errors (SSE) and Mean Squared Error (MSE).
SSE is the sum of the squared differences between the true and predicted values.
MSE is the mean of the squared differences between the true and predicted values.
"""
def compute_sse(y_true, y_pred):
    return np.sum((y_true - y_pred) ** 2)
 
def compute_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

In [12]:
for name, dataset in [('Diabetes', diabetes), ('California Housing', california)]:
    X_train, X_test, y_train, y_test = train_test_split(
        dataset.data, dataset.target, train_size=0.6, random_state=42
    )
    knn = KnnRegressor(k=5)
    knn.fit(X_train, y_train)

    train_sse = compute_sse(y_train, knn.predict(X_train))
    test_sse  = compute_sse(y_test,  knn.predict(X_test))
    train_mse = compute_mse(y_train, knn.predict(X_train))
    test_mse  = compute_mse(y_test,  knn.predict(X_test))
    train_rmse = np.sqrt(train_mse) # Root Mean Squared Error for the training set
    test_rmse  = np.sqrt(test_mse)  # Root Mean Squared Error for the test set
    print(f"{name} — Train SSE: {train_sse:.2f}, Test SSE: {test_sse:.2f}, Train MSE: {train_mse:.2f}, Test MSE: {test_mse:.2f}, Train RMSE: {train_rmse:.2f}, Test RMSE: {test_rmse:.2f}")


Diabetes — Train SSE: 714215.80, Test SSE: 627470.76, Train MSE: 2695.15, Test MSE: 3545.03, Train RMSE: 51.91, Test RMSE: 59.54
California Housing — Train SSE: 9551.60, Test SSE: 9613.81, Train MSE: 0.77, Test MSE: 1.16, Train RMSE: 0.88, Test RMSE: 1.08


---
## Question 2 – L-Fold Cross Validation
### Q2.I – Implement `LFold`
*GitHub issue: #6*

In [ ]:
nbr_of_folds = 5

class My_LFold:
    def __init__(self, lfolds=nbr_of_folds, shuffled=False, random_state=None):
        if not isinstance(lfolds, int) or lfolds < 2:
            raise ValueError(f"lfolds must be an integer >= 2, got {lfolds}")
        if not isinstance(shuffled, bool):
            raise TypeError(f"shuffled must be a bool, got {type(shuffled).__name__}")
        self.lfolds = lfolds
        self.shuffled = shuffled
        self.random_state = random_state

    def get_n_splits(self, x=None, y=None, groups=None):
        return self.lfolds

    def split(self, x, y=None, groups=None):
        N = len(x)
        if self.lfolds > N:
            raise ValueError(f"Cannot have lfolds={self.lfolds} folds with only {N} samples")
        indices = np.arange(N)
        if self.shuffled:
            rng = np.random.default_rng(self.random_state)
            rng.shuffle(indices)
        fold_sizes = np.full(self.lfolds, N // self.lfolds, dtype=int)
        fold_sizes[:N % self.lfolds] += 1
        current = 0
        for fold_size in fold_sizes:
            start, stop = current, current + fold_size
            test_idx = indices[start:stop]
            train_idx = np.concatenate([indices[:start], indices[stop:]])
            current = stop
            yield train_idx, test_idx

In [20]:
# Verify LFold correctness
for idx_train, idx_test in My_LFold(lfolds=nbr_of_folds).split(list(range(20))):
    print(idx_train, idx_test)

[ 4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19] [0 1 2 3]
[ 0  1  2  3  8  9 10 11 12 13 14 15 16 17 18 19] [4 5 6 7]
[ 0  1  2  3  4  5  6  7 12 13 14 15 16 17 18 19] [ 8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11 16 17 18 19] [12 13 14 15]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19]


### Q2.II – CV Experiment: K=1..50
*GitHub issues: #7, #8, #9*

In [ ]:
def run_cv_experiment(X, y, k_range=range(1, 51), lfolds=nbr_of_folds):
    """
    Run L-fold CV for each K and return mean/std of train and test MSE.
    Returns: dict with keys 'train_mean', 'train_std', 'test_mean', 'test_std'
    """
    results = {
        'train_mean': [],
        'train_std':  [],
        'test_mean':  [],
        'test_std':   [],
    }

    k_list = list(k_range)
    total = len(k_list)

    # Pre-compute splits once — same random_state means identical splits every time
    splits = list(My_LFold(lfolds=lfolds, shuffled=True, random_state=42).split(X))

    for i, k in enumerate(k_list):
        train_mse_list = []
        test_mse_list = []

        for train_idx, test_idx in splits:
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            knn = KnnRegressor(k=k)
            knn.fit(X_train, y_train)

            train_mse_list.append(compute_mse(y_train, knn.predict(X_train)))
            test_mse_list.append(compute_mse(y_test, knn.predict(X_test)))

        results['train_mean'].append(np.mean(train_mse_list))
        results['train_std'].append(np.std(train_mse_list))
        results['test_mean'].append(np.mean(test_mse_list))
        results['test_std'].append(np.std(test_mse_list))

        print(f"  K={k:>3}  [{i+1}/{total}]  {(i+1)/total*100:.0f}% complete", flush=True) if DEBUG else None

    return results


# Run on both datasets
results_diabetes = run_cv_experiment(diabetes.data, diabetes.target)
results_california = run_cv_experiment(california.data, california.target)

In [ ]:
print("Diabetes", results_diabetes) if DEBUG else None
print("California", results_california) if DEBUG else None


Diabetes {'train_mean': [np.float64(0.0), np.float64(1515.292780605304), np.float64(2024.6805743612726), np.float64(2262.553852471151), np.float64(2426.088515036571), np.float64(2561.436321708466), np.float64(2617.5741571070257), np.float64(2688.8917235339545), np.float64(2753.7064227327573), np.float64(2788.4948951681313), np.float64(2807.133761972731), np.float64(2830.097865551127), np.float64(2857.142768277296), np.float64(2856.27202221312), np.float64(2876.7315452911016), np.float64(2882.9373383008433), np.float64(2895.0246587682377), np.float64(2921.4321103294415), np.float64(2941.4122964361777), np.float64(2965.344042440902), np.float64(2977.744160367155), np.float64(2992.2059014766937), np.float64(3011.516418556458), np.float64(3023.360040580114), np.float64(3048.4642268209536), np.float64(3063.619843537761), np.float64(3079.2642470910605), np.float64(3088.4475274115985), np.float64(3102.436385550625), np.float64(3118.5908167371767), np.float64(3123.3020664596293), np.float64(31

### Q2.III – Plots with STE Error Bars
*GitHub issue: #10*

In [18]:
def plot_cv_results(results, title, lfolds=nbr_of_folds):
    """
    Plot mean train/test MSE vs K with STE error bars.
    STE = 1.96 * std / sqrt(L)
    """
    
    k_values = range(1, len(results['train_mean']) + 1)

    train_ste = 1.96 * results['train_std'] / np.sqrt(lfolds)
    test_ste = 1.96 * results['test_std'] / np.sqrt(lfolds)

    plt.figure(figsize=(10, 6))
    plt.errorbar(k_values, results['train_mean'], yerr=train_ste, label='Train MSE', fmt='-o', capsize=5)
    plt.errorbar(k_values, results['test_mean'], yerr=test_ste, label='Test MSE', fmt='-o', capsize=5)
    plt.title(title)
    plt.xlabel('K')
    plt.ylabel('MSE')
    plt.legend()
    plt.show()
    return plt

plot_cv_results(results_diabetes, 'Diabetes Dataset')
plot_cv_results(results_california, 'California Housing Dataset')

NameError: name 'nbr_of_folds' is not defined

### Discussion: Effect of K (Overfitting / Underfitting)
*GitHub issue: #11*

<!-- YOUR DISCUSSION HERE -->

### Discussion: Effect of L on CV Stability
*GitHub issue: #12*

<!-- YOUR DISCUSSION HERE -->

In [ ]:
# YOUR CODE HERE: repeat experiment with different values of L to support discussion

---
## Question 3 – Automatic Model Selection
### Q3.I – Implement `KnnRegressorCV`
*GitHub issues: #13, #14*

In [ ]:
class KnnRegressorCV(BaseEstimator):
    def __init__(self, ks=list(range(1, 21)), cv=LFold(5)):
        # YOUR CODE HERE
        pass

    def fit(self, x, y):
        # YOUR CODE HERE
        self.k_ = None  # best K selected by internal CV
        return self

    def predict(self, x):
        # YOUR CODE HERE
        pass

### Q3.II – Evaluate with Outer CV (Nested CV)
*GitHub issue: #15*

In [ ]:
# YOUR CODE HERE: evaluate KnnRegressorCV on both datasets
# Report self.k_ per outer fold and compare to best K from Q2

### Discussion: When Does Internal CV Succeed?
*GitHub issue: #16*

<!-- YOUR DISCUSSION HERE -->

---
## Tests
### LFold Test Suite

In [ ]:
def check_lfold_invariants(lfolds, n, label):
    """
    For a given lfolds and n, verify that every split satisfies:
      - no overlap between train and test indices
      - train + test = all N samples (no samples lost)
      - every sample appears in exactly one test fold across all folds
      - get_n_splits returns lfolds
    """
    data = list(range(n))
    splitter = My_LFold(lfolds=lfolds)
    assert splitter.get_n_splits(data) == lfolds, f"{label}: get_n_splits should return {lfolds}"

    all_test = []
    for fold, (train, test) in enumerate(splitter.split(data)):
        overlap = np.intersect1d(train, test)
        assert len(overlap) == 0, \
            f"{label} fold {fold}: train/test overlap on indices {overlap}"
        assert len(train) + len(test) == n, \
            f"{label} fold {fold}: train ({len(train)}) + test ({len(test)}) != {n}"
        all_test.extend(test.tolist())

    assert sorted(all_test) == list(range(n)), \
        f"{label}: not every sample appears in exactly one test fold"

    print(f"  PASS  {label}")


def check_fold_sizes(lfolds, n, label):
    """Verify that fold sizes differ by at most 1 (remainder distributed correctly)."""
    data = list(range(n))
    sizes = [len(test) for _, test in My_LFold(lfolds=lfolds).split(data)]
    assert max(sizes) - min(sizes) <= 1, \
        f"{label}: fold sizes differ by more than 1: {sizes}"
    print(f"  PASS  {label}")


def check_shuffle_reproducibility(n, label):
    """Verify that the same random_state yields identical splits on two calls."""
    data = list(range(n))
    splits_a = list(My_LFold(lfolds=5, shuffled=True, random_state=42).split(data))
    splits_b = list(My_LFold(lfolds=5, shuffled=True, random_state=42).split(data))
    for i, ((ta, tea), (tb, teb)) in enumerate(zip(splits_a, splits_b)):
        assert np.array_equal(ta, tb) and np.array_equal(tea, teb), \
            f"{label} fold {i}: splits differ across identical random_state"
    print(f"  PASS  {label}")


def check_shuffle_differs_from_no_shuffle(n, label):
    """Verify that shuffled splits are not identical to unshuffled splits."""
    data = list(range(n))
    unshuffled = [test.tolist() for _, test in My_LFold(lfolds=5, shuffled=False).split(data)]
    shuffled   = [test.tolist() for _, test in My_LFold(lfolds=5, shuffled=True, random_state=0).split(data)]
    assert unshuffled != shuffled, f"{label}: shuffled and unshuffled splits are identical"
    print(f"  PASS  {label}")


def check_raises(callable_, exc_type, label):
    """Verify that calling callable_ raises exc_type."""
    try:
        callable_()
        assert False, f"{label}: expected {exc_type.__name__} but nothing was raised"
    except exc_type:
        print(f"  PASS  {label}")


print("=== My_LFold invariant checks ===")
check_lfold_invariants(5, 20, "lfolds=5, n=20 (even split)")
check_lfold_invariants(5, 21, "lfolds=5, n=21 (uneven split, remainder=1)")
check_lfold_invariants(5, 23, "lfolds=5, n=23 (uneven split, remainder=3)")
check_lfold_invariants(2,  5, "lfolds=2, n=5  (minimum lfolds)")
check_lfold_invariants(5,  5, "lfolds=5, n=5  (lfolds == n, leave-one-out style)")

print("\n=== Fold size checks ===")
check_fold_sizes(5, 20, "lfolds=5, n=20 (even)")
check_fold_sizes(5, 21, "lfolds=5, n=21 (uneven)")
check_fold_sizes(3,  7, "lfolds=3, n=7  (uneven)")

print("\n=== Shuffle checks ===")
check_shuffle_reproducibility(20, "shuffled=True, same random_state → same splits")
check_shuffle_differs_from_no_shuffle(20, "shuffled=True vs False → different splits")

print("\n=== Warranty guard checks ===")
check_raises(lambda: My_LFold(lfolds=1),         ValueError, "lfolds=1 raises ValueError")
check_raises(lambda: My_LFold(lfolds=0),         ValueError, "lfolds=0 raises ValueError")
check_raises(lambda: My_LFold(lfolds=-1),        ValueError, "lfolds=-1 raises ValueError")
check_raises(lambda: My_LFold(lfolds=1.5),       ValueError, "lfolds=1.5 (float) raises ValueError")
check_raises(lambda: My_LFold(lfolds=2, shuffled="yes"), TypeError, "shuffled='yes' (str) raises TypeError")
check_raises(lambda: My_LFold(lfolds=2, shuffled=1),    TypeError, "shuffled=1 (int) raises TypeError")
check_raises(lambda: next(My_LFold(lfolds=10).split(list(range(5)))),
                                                  ValueError, "lfolds > n raises ValueError")

print("\nAll tests passed.")